# 30 — Matriz de features por locus → Drive

Toma los BAMs que ya están en `Drive/10_bam/` —solo llega ahí lo que pasó
`align.sh verificar`— y arma, por proyecto, **una fila por locus y una columna
por feature**:

1. **Anotar** con `yasma tradeoff` sobre nuestro BAM → `Drive/20_yasma/<org>/<rol>/`.
2. **Features** desde `loci.txt`, los reads del BAM y el genoma (pliegue con
   ViennaRNA) → `Drive/40_features/<org>/<rol>.matriz.tsv`.
3. **Etiquetar** por secuencia contra los maduros de MirGeneDB (`docs/positivos.md`).
4. **Exportar** lo que entra al entrenamiento —solo primarios de `gadmo`,
   `galga` y `maggi`— a `Drive/40_features/features_entrenamiento.tsv`, el
   formato que lee `scripts/loo_cv.py`.

Todo lo que decide está en `scripts/matriz.py` (con banco en
`tests/test_matriz.py`); las celdas solo mueven ficheros. Qué es cada feature y
por qué: `docs/matriz.md`.

**El duplicado nunca entra al entrenamiento**: es la validación independiente.
Su matriz se arma igual —para validar después— pero `exportar` lo deja afuera.

## Preámbulo: montar Drive y clonar el repo

Las mismas dos celdas que en los otros notebooks (`validate_notebooks.py` exige
que sean idénticas).

In [ ]:
from google.colab import drive
import os, pathlib

# drive.mount falla con un 'mount failed' pelado y nada mas. Lo que lo causa,
# casi siempre: en la ventana de permisos quedo una casilla sin tildar, se
# cerro, o se eligio otra cuenta; o quedo un montaje a medias de un intento
# anterior. El segundo intento desmonta y fuerza. Si tampoco, el error dice
# que hacer en vez de dejar un traceback de google.colab que no lo dice.
def _montar():
    try:
        drive.mount('/content/drive')
        return
    except Exception as e:
        print(f'primer intento: {e} -- desmonto y reintento')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
    except Exception as e:
        raise RuntimeError(
            f'No pude montar Drive ({e}). Casi siempre es la ventana de permisos:\n'
            '  1. Entorno de ejecucion -> Desconectar y borrar entorno de ejecucion\n'
            '  2. Corre esta celda de nuevo. En la ventana elegi seb.ugazm@gmail.com\n'
            '     y tilda TODAS las casillas (o "Seleccionar todo")\n'
            '  3. Si la ventana no aparece: habilita ventanas emergentes y cookies de\n'
            '     terceros para colab.research.google.com, o proba en incognito'
        ) from None


_montar()
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), (f'Drive monto, pero no veo {DRIVE}: casi seguro es otra cuenta. '
                        'Tiene que ser seb.ugazm@gmail.com -- borra el entorno y volve a montar.')
print('Drive OK:', DRIVE)


In [ ]:
import shutil, subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def _actualizar():
    # El clon es un CACHE del repo, no un espacio de trabajo: nada de lo que se
    # escribe durante una corrida vive adentro (el ledger va a Drive). Por eso
    # reset --hard y no pull --ff-only: el pull falla apenas un archivo
    # versionado quede modificado, y fallaba sin hacer ruido, asi que la celda
    # seguia corriendo con el codigo viejo.
    for args in (['fetch', '--depth', '1', 'origin', 'HEAD'],
                 ['reset', '--hard', 'FETCH_HEAD']):
        r = subprocess.run(['git', '-C', str(CLON)] + args,
                           capture_output=True, text=True)
        if r.returncode != 0:
            return False
    return True


def _clonar_de_cero():
    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


def clonar():
    if CLON.exists():
        if _actualizar():
            return 'clon actualizado'
        # Un clon que no se puede actualizar es peor que no tenerlo: la celda
        # seguiria con codigo viejo sin avisar. Se tira y se clona de nuevo.
        shutil.rmtree(CLON)
    return _clonar_de_cero()


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())


## Herramientas

YASMA pineado por ref de git (el `pyproject.toml` del tag dice otra versión),
ViennaRNA (sin él no arranca ningún subcomando de YASMA, y además pliega las
ventanas de la matriz) y pysam. Cada notebook corre en su propia VM: lo que
instaló `20_alinear` no existe acá.

Y **el parche de memoria de `tradeoff`**. Sin él, `tradeoff` reserva 8 bytes por
base del genoma y por librería antes de leer un read, y muere con `-9` en
cualquier genoma de cientos de Mb: gadmo, phypa y prupe murieron así. El parche
guarda lo mismo de forma dispersa y **no cambia la salida**
(`scripts/tradeoff_parche.py`); `matriz.anotar` se niega a correr sin él.


In [ ]:
if 'CLON' not in globals():
    raise RuntimeError('Falta el Preámbulo: corré a mano las celdas de montar Drive y '
                       'clonar el repo, y después esta.')

import shutil, subprocess, sys

YASMA_REF = 'git+https://github.com/NateyJay/YASMA@v1.1.1'

if not shutil.which('yasma'):
    r = subprocess.run(f'pip -q install ViennaRNA {YASMA_REF}', shell=True,
                       capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:])

for mod in ('pysam', 'RNA', 'numpy', 'pandas', 'sklearn', 'openpyxl'):
    __import__(mod)
r = subprocess.run(['yasma', '--help'], capture_output=True, text=True)
print(f"[{'OK ' if r.returncode == 0 else 'MAL'}] yasma")
if r.returncode != 0:
    raise RuntimeError('yasma no arranca: ' + (r.stdout + r.stderr)[-800:])
import pysam, RNA
print('pysam', pysam.__version__, '· ViennaRNA', RNA.__version__)

# EL PARCHE DE MEMORIA. Idempotente; se niega si tradeoff.py cambió.
r = subprocess.run([sys.executable, str(CLON / 'scripts' / 'tradeoff_parche.py')],
                   capture_output=True, text=True)
print((r.stdout + r.stderr).strip())
if r.returncode != 0:
    raise RuntimeError('el parche de memoria de tradeoff no se aplicó: sin él, -9 en '
                       'cualquier genoma de cientos de Mb')


## Configuración

El BAM se copia al disco de la VM antes de leerlo: `pysam` salta de locus en
locus, y sobre el montaje de Drive cada salto es un viaje de red. El genoma se
verifica contra `data/genomas.sha256` con `align.sh fasta`, que es el mismo
chequeo del alineamiento pero sin construir el índice de bowtie.

In [ ]:
if not all(n in globals() for n in ('CLON', 'DRIVE')):
    raise RuntimeError('Falta el Preámbulo: corré a mano las celdas de montar Drive y '
                       'clonar el repo, y después esta.')

import importlib, os, pathlib, shutil, subprocess, sys, time

TRABAJO = pathlib.Path('/content/matriz')     # disco efímero: BAM local y anotación
GENOMES = pathlib.Path('/content/genomes')    # el .fna verificado
for _d in (TRABAJO, GENOMES):
    _d.mkdir(parents=True, exist_ok=True)
YASMA_DRIVE   = DRIVE / '20_yasma'            # <org>/<rol>/tradeoff/loci.txt ...
FEAT_DRIVE    = DRIVE / '40_features'         # <org>/<rol>.matriz.tsv
MADUROS_DRIVE = FEAT_DRIVE / 'maduros'        # <org>.fa (MirGeneDB)
ORGANISMOS    = CLON / 'data' / 'organismos.tsv'

ENV = dict(os.environ, GENOMAS_LEDGER=str(CLON / 'data' / 'genomas.sha256'),
           GENOMES_DIR=str(GENOMES))

sys.path.insert(0, str(CLON / 'scripts'))
import matriz
importlib.reload(matriz)


def fasta_de(org):
    """El genoma verificado contra data/genomas.sha256, sin índice de bowtie.
    Copia de Drive solo lo que falta o cambió de tamaño: un genoma de 1 Gb no se
    baja dos veces."""
    src, dst = DRIVE / '70_genomas' / org, GENOMES / org
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        g = dst / f.name
        if not g.exists() or g.stat().st_size != f.stat().st_size:
            shutil.copy2(f, g)
    r = subprocess.run([str(CLON / 'scripts' / 'align.sh'), 'fasta', org],
                       capture_output=True, text=True, env=ENV)
    print(r.stderr.strip())
    if r.returncode != 0:
        raise RuntimeError(f'align.sh fasta {org} falló: no uso un genoma sin verificar')
    return pathlib.Path(r.stdout.strip())


print('trabajo :', TRABAJO)
print('drive   :', YASMA_DRIVE, '·', FEAT_DRIVE)

## 1. Qué hay

Un proyecto entra a la matriz cuando su BAM está en `Drive/10_bam/`. Para el
entrenamiento importan los **primarios** de `gadmo`, `galga` y `maggi`; el resto
se arma igual —los duplicados para validar, los de aplicación para predecir—.

In [ ]:
_faltan = [n for n in ('matriz', 'DRIVE', 'ORGANISMOS') if n not in globals()]
if _faltan:
    raise RuntimeError(f'Faltan {_faltan}. Corré a mano, en orden: Preámbulo, Herramientas, '
                       'Configuración — y después esta celda.')

EST = matriz.estado_drive(DRIVE, ORGANISMOS)
print(f"{'PROYECTO':22s} {'SET':14s} {'BAM GB':>7s}  {'ANOTACIÓN':10s} MATRIZ")
for e in EST:
    print(f"{e['org'] + '/' + e['rol']:22s} {e['set_modelo']:14s} {e['bam_gb']:7.2f}  "
          f"{'sí' if e['anotacion'] else '-':10s} {'sí' if e['matriz'] else '-'}")
PENDIENTES = [f"{e['org']}/{e['rol']}" for e in EST if not e['matriz']]
print()
print('pendientes:', PENDIENTES or 'ninguno')
_ent = [e for e in EST if e['set_modelo'] == 'entrenamiento' and e['rol'] == 'primario']
print(f"primarios de entrenamiento con BAM: {len(_ent)} de 3 "
      f"({', '.join(e['org'] for e in _ent) or 'ninguno'})")

## 1b. Control del parche (una vez)

Corre `tradeoff` **original y parchado** sobre el mismo BAM real y compara byte a
byte todo lo que escriben. En BAMs sintéticos dio igual (`tests/test_tradeoff_parche.py`);
esto lo comprueba sobre datos de verdad. Va sobre `sclsc/duplicado` porque es el
más chico (39 Mb, 2 librerías): **el original tiene que entrar en RAM**.

Usa un **tramo** del BAM, los reads de los primeros scaffolds hasta juntar
`MAX_READS`, y no los 32 M enteros. Con el BAM entero son dos `tradeoff`
completos: del orden de una hora sin imprimir nada, que en Colab parece una
celda colgada. Para probar que la salida es la misma alcanza con reads reales
—loci, multimapeados, librerías de distinta profundidad—, y eso está en
cualquier tramo.

Tarda unos minutos y avisa cada 30 s cuánto lleva. Tiene que terminar en
`>>> IGUALES`; imprime además la memoria pico de cada versión. Se corre una vez.


In [ ]:
CONTROL   = 'sclsc/duplicado'   # el más chico: el original tiene que entrar en RAM
MAX_READS = 3_000_000           # un tramo del BAM real; con el entero es ~1 h

_faltan = [n for n in ('CLON', 'DRIVE', 'TRABAJO') if n not in globals()]
if _faltan:
    raise RuntimeError(f'Faltan {_faltan}. Corré a mano, en orden: Preámbulo, Herramientas, '
                       'Configuración — y después esta celda.')

_org, _rol = CONTROL.split('/')
_cdir = TRABAJO / 'control'
_cdir.mkdir(parents=True, exist_ok=True)
for _suf in ('.bam', '.bam.bai'):
    shutil.copy2(DRIVE / '10_bam' / _org / f'{_rol}{_suf}', _cdir / f'{_rol}{_suf}')
# En vivo y no con capture_output: con la salida retenida hasta el final, una
# corrida de minutos se ve igual que una celda colgada.
_p = subprocess.Popen([sys.executable, '-u', str(CLON / 'scripts' / 'tradeoff_control.py'),
                       str(_cdir / f'{_rol}.bam'), '--max-reads', str(MAX_READS),
                       '--trabajo', str(_cdir / 'corridas')],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for _ln in _p.stdout:
    print(_ln, end='')
_rc = _p.wait()
shutil.rmtree(_cdir, ignore_errors=True)
if _rc != 0:
    raise RuntimeError('el control no dio IGUALES: no anotes con el parche hasta entender por qué')


## 2. Anotar y calcular las features

Por proyecto: BAM a la VM → genoma verificado → `yasma tradeoff` (si la
anotación ya está en Drive se reusa, salvo `REHACER = True`) → matriz → Drive.

`matriz.py features` cuenta los reads de cada locus con **la misma regla que
YASMA** y compara contra la columna `Reads` de `loci.txt`. Si no coinciden, no se
está mirando lo mismo que YASMA miró, y la matriz **no sube**.

Un proyecto que falla no frena a los demás; al final se listan.

In [ ]:
SOLO    = []      # vacío = los PENDIENTES de §1; o p. ej. ['gadmo/primario']
REHACER = False   # True = re-anota aunque la anotación ya esté en Drive

_faltan = [n for n in ('PENDIENTES', 'fasta_de', 'TRABAJO') if n not in globals()]
if _faltan:
    raise RuntimeError(f'Faltan {_faltan}. Corré a mano, en orden: Preámbulo, Herramientas, '
                       'Configuración y §1 — y después esta celda.')

hechos, fallados = [], []
for p in (SOLO or PENDIENTES):
    org, rol = p.split('/')
    t0 = time.monotonic()
    local = TRABAJO / f'{org}_{rol}'
    local.mkdir(parents=True, exist_ok=True)
    bam = local / f'{rol}.bam'
    print(f'\n=== {p}')
    try:
        for suf in ('.bam', '.bam.bai'):
            shutil.copy2(DRIVE / '10_bam' / org / f'{rol}{suf}', local / f'{rol}{suf}')
        fna = fasta_de(org)
        anot, en_drive = local / 'anot', YASMA_DRIVE / org / rol
        if (en_drive / 'tradeoff' / 'loci.txt').exists() and not REHACER:
            print('anotación: ya está en Drive, la reuso')
            shutil.copytree(en_drive, anot, dirs_exist_ok=True)
        else:
            matriz.anotar(bam, anot)
            shutil.copytree(anot, en_drive, dirs_exist_ok=True)
            print(f'anotación -> {en_drive}')
        salida = local / f'{rol}.matriz.tsv'
        rc = matriz.main(['features', str(anot), str(bam), '--org', org, '--rol', rol,
                          '--genoma', str(fna), '-o', str(salida)])
        if rc != 0:
            raise RuntimeError('los reads por locus no coinciden con loci.txt: no subo esta matriz')
        (FEAT_DRIVE / org).mkdir(parents=True, exist_ok=True)
        shutil.copy2(salida, FEAT_DRIVE / org / salida.name)
        hechos.append((p, (time.monotonic() - t0) / 60))
        print(f'matriz -> {FEAT_DRIVE / org / salida.name}  ({hechos[-1][1]:.0f} min)')
    except (Exception, SystemExit) as e:
        fallados.append((p, str(e)))
        print(f'FALLÓ {p}: {e}')
    finally:
        for f in local.glob('*.bam*'):
            f.unlink()

print('\n--- resumen ---')
for p, m in hechos:
    print(f'   ok     {p}  ({m:.0f} min)')
for p, e in fallados:
    print(f'   FALLÓ  {p}: {e}')

## 3. Los maduros de MirGeneDB

El conjunto positivo de los tres organismos de entrenamiento (`docs/positivos.md`).
Se bajan una vez a `Drive/40_features/maduros/<org>.fa` y se validan: una URL que
cambió devuelve una página HTML con código 200, y etiquetar contra eso daría
**todo sin etiquetar, sin ningún error**.

Los códigos de especie y las URLs están **a confirmar en la primera corrida**
(esta sesión de desarrollo no llega a mirgenedb.org). Si ninguna URL da un FASTA
de maduros, la celda dice cómo bajarlo a mano. `maggi` en MirGeneDB figura como
*Crassostrea gigas*.

In [ ]:
_faltan = [n for n in ('matriz', 'MADUROS_DRIVE') if n not in globals()]
if _faltan:
    raise RuntimeError(f'Faltan {_faltan}. Corré a mano, en orden: Preámbulo, Herramientas, '
                       'Configuración — y después esta celda.')

import hashlib, urllib.request

MIRGENEDB = {'gadmo': 'gmo', 'galga': 'gga', 'maggi': 'cgi'}
URLS = ('https://mirgenedb.org/fasta/{sp}?mat=1',
        'https://mirgenedb.org/static/data/{sp}/{sp}-mat.fas')

MADUROS_DRIVE.mkdir(parents=True, exist_ok=True)
MADUROS, MADUROS_URL = {}, {}
for org, sp in MIRGENEDB.items():
    dest = MADUROS_DRIVE / f'{org}.fa'
    if dest.exists():
        n, mot = matriz.validar_maduros(dest.read_text())
        if mot is None:
            MADUROS[org], MADUROS_URL[org] = dest, 'ya en Drive'
            print(f'{org}: {n} maduros, ya en Drive')
            continue
        print(f'{org}: {dest.name} está pero no sirve ({mot}); lo vuelvo a bajar')
    for u in URLS:
        url = u.format(sp=sp)
        try:
            txt = urllib.request.urlopen(url, timeout=60).read().decode('utf-8', 'replace')
        except Exception as e:
            print(f'   {url}: {e}')
            continue
        n, mot = matriz.validar_maduros(txt)
        if mot:
            # Lo que devolvió de verdad: una página de error, un FASTA vacío o
            # algo inesperado se distinguen acá y no adivinando.
            print(f'   {url}: {mot}\n      devolvió {len(txt)} caracteres; empieza con: {txt[:160]!r}')
            continue
        dest.write_text(txt)
        MADUROS[org], MADUROS_URL[org] = dest, url
        print(f'{org}: {n} maduros de {url}')
        break
    else:
        print(f'{org}: NO pude bajar los maduros ({sp}). A mano: en https://mirgenedb.org/browse '
              f'buscá la especie (maggi puede figurar como Crassostrea o como Magallana gigas), '
              f'bajá el FASTA de MADUROS y subilo a {dest}. Si no está en MirGeneDB, '
              f'avisá: maggi es organismo de entrenamiento y eso cambia el diseño.')

## 4. Etiquetar y exportar lo que entra al entrenamiento

`etiquetar` agrega `y` (1 = coincide con un maduro, 0 = **sin etiquetar**, no
negativo) y la mejor coincidencia con sus diferencias (`y_ref`, `y_mm`,
`y_dlen`), así que medir la sensibilidad al umbral es releer la tabla.

`exportar` saca **solo los primarios de entrenamiento** a
`features_entrenamiento.tsv` y dice qué quedó afuera y por qué. Con tres
organismos de entrenamiento se corre la validación cruzada (`loo_cv.py`); con
menos, todavía no mide transferencia y la celda lo dice.

In [ ]:
_faltan = [n for n in ('matriz', 'MADUROS', 'FEAT_DRIVE', 'TRABAJO') if n not in globals()]
if _faltan:
    raise RuntimeError(f'Faltan {_faltan}. Corré a mano, en orden: Preámbulo, Herramientas, '
                       'Configuración y §3 — y después esta celda.')

etiquetadas = []
for e in matriz.estado_drive(DRIVE, ORGANISMOS):
    if not e['matriz'] or e['org'] not in MADUROS:
        continue
    m = FEAT_DRIVE / e['org'] / f"{e['rol']}.matriz.tsv"
    loc = TRABAJO / f"{e['org']}_{e['rol']}.matriz.tsv"
    shutil.copy2(m, loc)
    print(f"--- {e['org']}/{e['rol']}")
    matriz.main(['etiquetar', str(loc), str(MADUROS[e['org']]), '-o', str(loc)])
    shutil.copy2(loc, m)
    etiquetadas.append(loc)

ENTRENAMIENTO = FEAT_DRIVE / 'features_entrenamiento.tsv'
if not etiquetadas:
    print('ninguna matriz de entrenamiento etiquetada todavía (¿§2 y §3?)')
else:
    try:
        matriz.main(['exportar', *map(str, etiquetadas), '-o', str(TRABAJO / ENTRENAMIENTO.name)])
        shutil.copy2(TRABAJO / ENTRENAMIENTO.name, ENTRENAMIENTO)
        print(f'-> {ENTRENAMIENTO}')
        import csv
        with open(ENTRENAMIENTO) as _h:
            _orgs = len({f['org'] for f in csv.DictReader(_h, delimiter='\t')})
        if _orgs >= 3:
            subprocess.run([sys.executable, str(CLON / 'scripts' / 'loo_cv.py'), str(ENTRENAMIENTO)])
        else:
            print(f'{_orgs} organismo(s) de entrenamiento: loo_cv.py necesita los 3 '
                  '(gadmo, galga, maggi) para dejar uno afuera.')
    except SystemExit as e:
        print(f'exportar no armó nada: {e}')

## 5. Registrar en git qué matriz se armó

Lo que va a git es texto chico y necesario para reproducir:
`data/matrices.tsv` (por proyecto: loci, positivos, features y el **sha256** del
fichero de Drive) y `data/maduros.tsv` (de dónde salieron los positivos y su
sha256). Las matrices viven en Drive; el sha256 es el ancla de cuál se usó.

`REVISAR_PRIMERO = True` imprime el diff y no empuja. Para que llegue a git hay
que ponerlo en `False`, con un `GITHUB_TOKEN` de escritura en los Secrets.

In [ ]:
REVISAR_PRIMERO = True     # False = commitea y empuja

_faltan = [n for n in ('matriz', 'FEAT_DRIVE', 'MADUROS_DRIVE') if n not in globals()]
if _faltan:
    raise RuntimeError(f'Faltan {_faltan}. Corré a mano, en orden: Preámbulo, Herramientas, '
                       'Configuración — y después esta celda.')

import csv, datetime, hashlib, colab_git


def _fecha(p):
    return datetime.datetime.fromtimestamp(p.stat().st_mtime, datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')


filas = []
for e in matriz.estado_drive(DRIVE, ORGANISMOS):
    if e['matriz']:
        m = FEAT_DRIVE / e['org'] / f"{e['rol']}.matriz.tsv"
        filas.append({'proyecto': f"{e['org']}_{e['rol']}", 'org': e['org'], 'rol': e['rol'],
                      **matriz.resumen_matriz(m), 'fecha_utc': _fecha(m)})
mad = []
for f in sorted(MADUROS_DRIVE.glob('*.fa')):
    n, mot = matriz.validar_maduros(f.read_text())
    mad.append({'org': f.stem, 'fuente': 'MirGeneDB', 'url': globals().get('MADUROS_URL', {}).get(f.stem, ''),
                'secuencias': n, 'valido': 'si' if mot is None else 'no',
                'sha256': hashlib.sha256(f.read_bytes()).hexdigest(), 'fecha_utc': _fecha(f)})


def _escribir(ruta, filas, cols):
    viejo = ruta.read_text().splitlines() if ruta.exists() else []
    with open(ruta, 'w', newline='') as h:
        for l in viejo:
            if l.startswith('#'):
                h.write(l + '\n')
        w = csv.DictWriter(h, fieldnames=cols, delimiter='\t')
        w.writeheader()
        w.writerows(filas)


_escribir(CLON / 'data' / 'matrices.tsv', filas,
          ['proyecto', 'org', 'rol', 'loci', 'positivos', 'features', 'sha256', 'fecha_utc'])
_escribir(CLON / 'data' / 'maduros.tsv', mad,
          ['org', 'fuente', 'url', 'secuencias', 'valido', 'sha256', 'fecha_utc'])
print(colab_git.empujar(
    CLON, ['data/matrices.tsv', 'data/maduros.tsv'],
    f'matriz: {len(filas)} proyecto(s) desde Colab',
    revisar=REVISAR_PRIMERO,
    autor=('Colab', 'colab@users.noreply.github.com')))

## 6. Las features en Excel

Junta todas las matrices que hay en Drive en un solo `.xlsx`, para mirarlas:
- **Resumen**: proyecto, `set_modelo`, loci y positivos. Los conteos son
  fórmulas sobre cada hoja y se calculan al abrir el archivo.
- **Diccionario**: qué es cada columna, de dónde sale y por qué está.
- **Una hoja por proyecto**: una fila por locus, con filtros y la cabecera
  fija.

Si §4 ya etiquetó, trae la columna `y`. Recordá que `y = 0` es **sin
etiquetar**, no negativo.

Es data, no código: va a `Drive/40_features/features.xlsx` y no a git.
Volvé a correr esta celda cada vez que se agregue una matriz.


In [ ]:
_faltan = [n for n in ('matriz', 'FEAT_DRIVE', 'TRABAJO', 'ORGANISMOS') if n not in globals()]
if _faltan:
    raise RuntimeError(f'Faltan {_faltan}. Corré a mano, en orden: Preámbulo, Herramientas, '
                       'Configuración — y después esta celda.')

EXCEL = FEAT_DRIVE / 'features.xlsx'
_ms = []
for _e in matriz.estado_drive(DRIVE, ORGANISMOS):
    if _e['matriz']:
        _loc = TRABAJO / 'excel' / f"{_e['org']}_{_e['rol']}.matriz.tsv"
        _loc.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(FEAT_DRIVE / _e['org'] / f"{_e['rol']}.matriz.tsv", _loc)
        _ms.append(_loc)
if not _ms:
    print('Todavía no hay matrices en Drive: corré §2.')
else:
    _out = TRABAJO / 'features.xlsx'
    matriz.main(['excel', *map(str, _ms), '-o', str(_out)])
    shutil.copy2(_out, EXCEL)
    print(f'-> {EXCEL}  ({EXCEL.stat().st_size / 1e6:.1f} MB)')
    print('Abrilo con Google Sheets o Excel: el Resumen se calcula al abrir.')
